# 04 - Medium-Term Modeling

This notebook reviews the one-month daily forecast model. It reads the saved daily metrics, validation predictions, final forecasts, and model file created by the pipeline.

The daily task is different from the half-hourly task. We are forecasting one value per day for each ACORN segment, so the model needs to capture broader weather and weekly patterns rather than within-day peaks.


## Setup

We load the saved daily artifacts. The notebook does not retrain the model, which keeps the results consistent with the exported prediction files.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass


import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

try:
    from IPython.display import display
except ImportError:
    display = None


def show_plot(fig):
    if display is not None:
        display(fig)
    plt.close(fig)

metrics_path = ROOT / "outputs" / "group5" / "metrics" / "validation_metrics.csv"
valid_path = ROOT / "outputs" / "group5" / "metrics" / "validation_predictions_daily.csv"
forecast_path = ROOT / "outputs" / "group5" / "predictions" / "group_5_daily_predict.csv"
model_path = ROOT / "models" / "medium_term" / "group5_daily_selected.joblib"

print("Run the pipeline first if these files are missing:")
print("EI-climat/bin/python scripts/group5_run_pipeline.py")

## Validation metrics

The table compares baselines and learned models on a later historical validation period. This is the same time-aware principle used for the short-term model.

For the daily horizon, we care about whether the model improves on recent-history baselines without overfitting the smaller daily dataset.


In [ ]:
metrics = pd.read_csv(metrics_path)
daily_metrics = metrics[metrics["frequency"] == "daily"].sort_values(["acorn", "rmse"])
daily_metrics

The selected medium-term model is the ridge regression model. It performs best on the validation split after the current preprocessing choices.

This is a good fit for the daily task because the dataset has fewer rows than the half-hourly task. A regularized linear model can be more stable when the signal is broad and the sample size is smaller.


## Validation curve for one segment

This plot compares actual and predicted daily consumption for ACORN-E on the validation period. It checks whether the model follows the direction and level of the daily series.


In [ ]:
valid = pd.read_csv(valid_path, parse_dates=["timestamp"])
acorn = "ACORN-E"
plot_df = valid[valid["Acorn"] == acorn].copy()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(plot_df["timestamp"], plot_df["actual"], marker="o", label="actual")
ax.plot(plot_df["timestamp"], plot_df["gradient_boosting"], marker="o", label="gradient_boosting")
ax.plot(plot_df["timestamp"], plot_df["previous_week"], marker="o", label="previous_week", alpha=0.7)
ax.set_title(f"Daily validation - {acorn}")
ax.set_ylabel("kWh")
ax.legend()
show_plot(fig)

The daily model should track the main movement of the series, but it will not match every day exactly. That is acceptable for this horizon as long as the forecast level and trend are realistic.

Large systematic gaps would be a warning that weather, holiday, or lag features are missing. The validation plot helps catch that kind of problem.


## Final one-month forecast

The final daily forecast covers 2014-01-13 through 2014-02-13, following the assignment template. We plot all three segments to compare their expected levels over the month.


In [ ]:
forecast = pd.read_csv(forecast_path, parse_dates=["Date"])
fig, ax = plt.subplots(figsize=(13, 5))
sns.lineplot(data=forecast, x="Date", y="Conso_kWh_predict", hue="Acorn", marker="o", ax=ax)
ax.set_title("Final one-month daily forecast")
show_plot(fig)

forecast.head()

The final forecast keeps the same segment ordering seen in the historical data: ACORN-E is highest, ACORN-F is in the middle, and ACORN-Q is lowest.

The month is in winter, so temperature remains important. The forecast should be read as a daily planning estimate, not as a replacement for measured consumption.


## Saved model check

The last cell confirms that the medium-term model file exists. The exported daily predictions and the dashboard depend on this artifact.


In [ ]:
print("Saved medium-term model:", model_path)
print("Exists:", model_path.exists())

The saved model path should exist after the pipeline has run. If the file is missing, rerun the pipeline so the dashboard and report read the current model outputs.
